# 論文 docx → J-STAGE 全文 XML 変換（全国数学教育学会誌『数学教育学研究』）

**ツールの版: 0.12.1**（試用版。正式版は 1.0 から）　… 手順 1 を実行すると、実際に読み込まれた版（とコミット ID）も表示されます。

このノートブックは、テンプレートで作成された論文の Word ファイル（.docx）を、J-STAGE の全文 XML（JATS 1.1）に変換します。
パソコンに何かをインストールする必要はありません。**上から順に、各セルの左の ▶ ボタンを押して実行してください。**
前の手順を実行せずに後の手順の ▶ を押すと、「先に手順 ○ を実行してください」と表示されて止まります。

| 手順 | 内容 |
|---|---|
| 1 | 準備（最初に 1 回だけ。▶ を押したら、終わるのを待たずに手順 2 の ▶ を押しておくと、準備の後に続けて実行されます） |
| 2 | 論文の docx と PDF をアップロード（ボタンが別々に出ます） |
| 3 | 書誌情報（巻・号・ページ・査読の有無・キーワード・日付）を入力 |
| 4 | AI による補完の方法を選ぶ |
| 5 | 変換を実行（結果のレポートが表示されます） |
| 6 | （手動モードのときだけ）AI の回答を貼り付けて再変換 |
| 7 | 結果をダウンロード |

- 本文の文字は 1 文字も変えません。変換後に、docx と XML の文字を 1 文字ずつ照合して確かめます（レポートの「文字の不変性チェック」）。
- アップロードしたファイルは、このノートブックを閉じる（ランタイムが切れる）と Google のサーバーから消えます。
- 「API で自動実行」を選ぶと、論文の本文が Anthropic 社の Claude API に送信されます。

ソースコードと説明書: https://github.com/thjpme/jstgxmlcng_JASME

In [ ]:
#@title 1. 準備（最初に 1 回だけ。1 分ほどかかります）
#@markdown **EMF/WMF の図を変換する（CONVERT_EMF）**: 図が Word の図形（EMF/WMF 形式）で貼られている論文では、画像の変換に LibreOffice を使います。チェックを入れると LibreOffice を**裏で**インストールします（数分かかりますが、終わるのを待たずに手順 2 以降へ進めます。手順 5 の変換は、インストールが終わってから始まります）。
CONVERT_EMF = True  #@param {type:"boolean"}
#@markdown **開発版を使う（USE_DEV_VERSION）**: 開発者向け。開発中の版を試すときだけチェックを入れます（実行するとブランチ名の入力欄が出ます）。
USE_DEV_VERSION = False  #@param {type:"boolean"}

import shutil, subprocess, sys
# このランタイムで一度ツールを使っていると、古い版がメモリに残ったままになる（入れ直しても反映されない）
loaded_before = "jasme_jats" in sys.modules
BRANCH = "main"
if USE_DEV_VERSION:
    BRANCH = input("開発版のブランチ名を入力して Enter を押してください: ").strip()
    if not BRANCH:
        raise SystemExit("ブランチ名が入力されていません。開発版を使わないときは、チェックを外して実行してください。")
url = f"git+https://github.com/thjpme/jstgxmlcng_JASME.git@{BRANCH}"
for args in (["jasme-jats[ai,images] @ " + url],                          # ツールと必要なライブラリ
             ["--force-reinstall", "--no-deps", "jasme-jats @ " + url]):  # ツール本体は毎回最新に入れ直す
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise SystemExit("ツールのインストールに失敗しました。" + ("ブランチ名を確認してください。" if USE_DEV_VERSION else
                                                                  "しばらくしてからもう一度実行してください。"))
if loaded_before:
    raise SystemExit("ツールを最新の版に入れ替えましたが、このランタイムには前の版が読み込まれたままです。"
                     "メニューの「ランタイム」→「セッションを再起動」（表示によっては「ランタイムを再起動」）を選んでから、"
                     "手順 1 からもう一度実行してください。")
# LibreOffice はインストールに数分かかるので裏で入れる（手順 5 で終わるのを待つ）
LO_LOG = "/content/libreoffice_install.log"
if CONVERT_EMF and not (shutil.which("soffice") or shutil.which("libreoffice")):
    if globals().get("LO_PROC") is None or LO_PROC.poll() is not None:
        LO_PROC = subprocess.Popen("apt-get -qq update && (apt-get -qq install -y --no-install-recommends "
                                   "libreoffice-draw-nogui || apt-get -qq install -y libreoffice-draw)",
                                   shell=True, stdout=open(LO_LOG, "w"), stderr=subprocess.STDOUT)
# 版の表示は別のプロセスで行う（ここで読み込むと、次に手順 1 を実行したときに「前の版が残っている」扱いになるため）
ver = subprocess.run([sys.executable, "-c", "from jasme_jats.pipeline import tool_version; print(tool_version())"],
                     capture_output=True, text=True).stdout.strip()
print("準備ができました。ツールの版:", ver)
if USE_DEV_VERSION:
    print("※ 開発版（ブランチ", BRANCH, "）を使っています。")
elif ver and not ver.startswith("0.12.1"):
    print("※ このノートブックの説明は版 0.12.1 のものです。ノートブックを開き直してください。")
if globals().get("LO_PROC") is not None and LO_PROC.poll() is None:
    print("LibreOffice を裏でインストールしています（数分）。終わるのを待たずに手順 2 へ進んでください。")
DONE = {1}

In [ ]:
#@title 2. 論文の docx と PDF をアップロード
#@markdown 実行すると、まず **docx 用**の「ファイル選択」ボタンが出ます。docx を選ぶと、続けて **PDF 用**の「ファイル選択」ボタンが出ます。
#@markdown **PDF もアップロードする（UPLOAD_PDF）**: J-STAGE に登載する PDF がまだなければ、チェックを外してから実行してください（PDF 用のボタンは出ません。後で jasme.zip に入れることもできます）。
UPLOAD_PDF = True  #@param {type:"boolean"}
_need = [n for n in (1,) if n not in globals().get("DONE", set())]
if _need:
    raise SystemExit("先に手順 " + "・".join(map(str, _need)) + " を実行してください（上から順に ▶ を押します。"
                     "前の手順をやり直したときは、その後の手順もやり直してください）。")

from pathlib import Path
from google.colab import files

WORK = Path("/content/jasme_work")
INPUT = WORK / "input"
INPUT.mkdir(parents=True, exist_ok=True)


def upload(what, ext):
    print(f"▼ {what}（{ext}）を選んでください")
    paths = []
    for name, data in files.upload().items():
        if not name.lower().endswith(ext):
            print(f"  「{name}」は {ext} ではないので使いません")
            continue
        (INPUT / name).write_bytes(data)
        paths.append(INPUT / name)
    return paths


docx_files = upload("論文の Word ファイル", ".docx")
if not docx_files:
    raise SystemExit("docx ファイルが選ばれていません。もう一度この手順を実行してください。")
if len(docx_files) > 1:
    print("  docx が複数選ばれたため、最初の 1 つだけを使います")
docx_path = docx_files[0]
pdf_path = None
if UPLOAD_PDF:
    print()
    pdf_files = upload("J-STAGE に登載する PDF", ".pdf")
    pdf_path = pdf_files[0] if pdf_files else None
print()
print("論文:", docx_path.name)
print("PDF :", pdf_path.name if pdf_path else "（なし。J-STAGE への登載には PDF が必要です）")
DONE = {n for n in globals().get("DONE", set()) if n < 2} | {2}

In [ ]:
#@title 3. 書誌情報を入力
#@markdown 各欄の上の説明（日本語）に対応する欄に入力します。日付は YYYY-MM-DD の形で、分からなければ空欄にします。
#@markdown ***
#@markdown **巻（volume）**（J-STAGE のフォルダ名・ファイル名と DOI に使います）
volume = ""  #@param {type:"string"}
#@markdown **号（issue）**
issue = ""  #@param {type:"string"}
#@markdown **最初のページ（first_page）**
first_page = ""  #@param {type:"string"}
#@markdown **最後のページ（last_page）**
last_page = ""  #@param {type:"string"}
#@markdown ***
#@markdown **査読の有無（peer_review）**（必須。必ず選んでください）
peer_review = "（選んでください）"  #@param ["（選んでください）", "査読あり", "査読なし"]
#@markdown **最終査読日（last_review_date）**（査読ありで、分かれば）
last_review_date = ""  #@param {type:"string"}
#@markdown ***
#@markdown **キーワード（日本語）（keywords_ja）**（任意。「；」または「;」で区切る。例: `数学教育；ジェンダー`）
keywords_ja = ""  #@param {type:"string"}
#@markdown **キーワード（英語）（keywords_en）**（任意。「;」または「；」で区切る。例: `equity; gender`）。日・英あわせて 100 個まで
keywords_en = ""  #@param {type:"string"}
#@markdown ***
#@markdown **発行日（pub_date）**
pub_date = ""  #@param {type:"string"}
#@markdown **J-STAGE 公開日（jstage_release_date）**
jstage_release_date = ""  #@param {type:"string"}
#@markdown **受付日（received）**
received = ""  #@param {type:"string"}
#@markdown **受理日（accepted）**
accepted = ""  #@param {type:"string"}
#@markdown ***
#@markdown **DOI（doi）**（空欄なら 10.24529/jasme.巻.号_開始ページ を使います）
doi = ""  #@param {type:"string"}
#@markdown **原稿種別（manuscript_type）**
manuscript_type = "論文 / Article"  #@param ["論文 / Article", "共同研究 / Collaborative Research Report", "講演記事 / Oration", "その他"]
#@markdown **原稿種別（日本語）（subject_other）**・**原稿種別（英語）（subject_other_en）**: 原稿種別で「その他」を選んだときだけ入力します
subject_other = ""  #@param {type:"string"}
subject_other_en = ""  #@param {type:"string"}
#@markdown **責任著者が何番目の著者か（corresponding_author）**（例: 1。複数なら「1,2」）
corresponding_author = ""  #@param {type:"string"}
_need = [n for n in (1, 2,) if n not in globals().get("DONE", set())]
if _need:
    raise SystemExit("先に手順 " + "・".join(map(str, _need)) + " を実行してください（上から順に ▶ を押します。"
                     "前の手順をやり直したときは、その後の手順もやり直してください）。")
if peer_review not in ("査読あり", "査読なし"):
    raise SystemExit("査読の有無（peer_review）を選んでください。")
if peer_review == "査読なし" and last_review_date.strip():
    raise SystemExit("査読なしのときは、最終査読日（last_review_date）を空欄にしてください。")
if manuscript_type == "その他":
    if not subject_other.strip() or not subject_other_en.strip():
        raise SystemExit("原稿種別で「その他」を選んだときは、原稿種別（日本語）（subject_other）と（英語）（subject_other_en）を入力してください。")
    subject_ja, subject_en = subject_other.strip(), subject_other_en.strip()
else:
    subject_ja, subject_en = manuscript_type.split(" / ", 1)
    if subject_other.strip() or subject_other_en.strip():
        print("※ 原稿種別は「" + manuscript_type + "」を使います（subject_other の入力は使いません）")
meta_input = {"volume": volume, "issue": issue, "fpage": first_page, "lpage": last_page,
              "peer_review": peer_review, "approved": last_review_date,
              "keywords_ja": keywords_ja, "keywords_en": keywords_en,
              "pub_date": pub_date, "epub_date": jstage_release_date, "received": received,
              "accepted": accepted, "doi": doi,
              "subject_ja": subject_ja, "subject_en": subject_en,
              "corresp": corresponding_author}
from jasme_jats.config import ArticleMeta, keyword_language_mismatch, keyword_problem
try:
    _meta = ArticleMeta.from_dict(meta_input)   # 日付の書き方などをここで確かめる
except ValueError as e:
    raise SystemExit(f"入力を確認してください: {e}")
if keyword_problem(_meta):
    raise SystemExit(keyword_problem(_meta) + "。")
print("入力しました（原稿種別: " + subject_ja + " / " + subject_en + "、" + peer_review + "）。未入力の項目はレポートに表示されます。")
print("キーワード（日本語）:", "；".join(_meta.keywords_ja) or "（なし）")
print("キーワード（英語）  :", "; ".join(_meta.keywords_en) or "（なし）")
if keyword_language_mismatch(_meta):
    print("※ 日本語・英語の欄を取り違えていないか確認してください:", "、".join(keyword_language_mismatch(_meta)))
DONE = {n for n in globals().get("DONE", set()) if n < 3} | {3}

In [ ]:
#@title 4. AI による補完の方法を選ぶ
#@markdown - **使わない**（推奨しません）: 機械的な変換だけを行います。取りこぼしがあっても補われません。テキストボックスなどに入った図表で、位置を機械的に決められないものがあると、手順 5 で変換を中止します（AI による補完が必要です）。
#@markdown - **API で自動実行**: Claude API を呼び出して、機械処理で取りこぼした引用・スタイルの誤り・画像の数式などを補完します。左の 🔑（シークレット）に名前 `ANTHROPIC_API_KEY` で API キーを登録し、「ノートブックからのアクセス」をオンにしてください（API の利用料がかかります）。キーの作り方と登録の手順は、このセルのすぐ下の「**API キーについての詳細**」を開いてください。
#@markdown - **手動**: AI への依頼文を作ります。claude.ai などのチャットに貼り付け、返ってきた回答を手順 6 で貼り戻します（API キー不要）。
#@markdown ***
#@markdown **AI による補完の方法（AI_MODE）**
AI_MODE = "手動（チャットに貼り付け）"  #@param ["使わない", "API で自動実行", "手動（チャットに貼り付け）"]
#@markdown **API で使うモデル（MODEL）**（通常は変更不要。「API キーについての詳細」の 5. を参照）
MODEL = "claude-opus-5-5"  #@param ["claude-opus-5-5", "自動（最新の Opus）"] {allow-input: true}
#@markdown **図表番号を分ける（SPLIT_LABELS）**: 図表番号（「図1」）を J-STAGE の <label> 要素に分けるか（分けると番号とタイトルの間の空白 1 文字を落とします。既定は分けない）
SPLIT_LABELS = False  #@param {type:"boolean"}
_need = [n for n in (1, 2, 3,) if n not in globals().get("DONE", set())]
if _need:
    raise SystemExit("先に手順 " + "・".join(map(str, _need)) + " を実行してください（上から順に ▶ を押します。"
                     "前の手順をやり直したときは、その後の手順もやり直してください）。")
AI = {"使わない": "none", "API で自動実行": "api", "手動（チャットに貼り付け）": "manual"}[AI_MODE]
print("AI による補完:", AI_MODE)
if AI == "none":
    from IPython.display import HTML, display
    from jasme_jats.colab import no_ai_box
    display(HTML(no_ai_box()))
DONE = {n for n in globals().get("DONE", set()) if n < 4} | {4}

<details>
<summary><b>API キーについての詳細（クリックで開きます）</b></summary>

<p>「API で自動実行」を使うときだけ必要です。手動モード（チャットに貼り付け）なら API キーは要りません。</p>

<h4>1. API キーを作る（最初に 1 回だけ）</h4>
<ol>
<li>Claude Console（<a href="https://platform.claude.com/" target="_blank">https://platform.claude.com/</a>）にログインします。アカウントがなければ作成します。
API の料金は、claude.ai の有料プラン（Pro など）とは別にかかります。</li>
<li>「Settings → Billing」（<a href="https://platform.claude.com/settings/billing" target="_blank">https://platform.claude.com/settings/billing</a>）で、利用料（クレジット）を購入します。
同じ画面で、1 か月に使う金額の上限を設定できます。使いすぎを防ぐため、上限を設定しておくことをお勧めします。</li>
<li>「Settings → API keys」（<a href="https://platform.claude.com/settings/keys" target="_blank">https://platform.claude.com/settings/keys</a>）で <b>Create key</b> を押し、名前（例: <code>jasme-colab</code>）を付けて作成します。</li>
<li>表示されたキー（<code>sk-ant-</code> で始まる長い文字列）をコピーします。<b>キーが表示されるのはこのときだけです。</b>
控えずに閉じてしまったら、新しいキーを作り直してください。</li>
</ol>

<h4>2. Colab にキーを登録する（Google アカウントごとに 1 回）</h4>
<ol>
<li>画面左端の 🔑（鍵の形）のアイコンを押します。「シークレット」という画面が開きます。</li>
<li>「<b>＋ 新しいシークレットを追加</b>」を押します。「ノートブックからのアクセス」「名前」「値」「アクション」の欄が並んだ行が 1 行増えます。</li>
<li>「<b>名前</b>」の欄に <code>ANTHROPIC_API_KEY</code> と入力します（大文字・小文字もこのとおりに。スペースは入れられません）。</li>
<li>「<b>値</b>」の欄に、コピーしておいた API キー（<code>sk-ant-</code> で始まる文字列）を貼り付けます。
入力した値は隠して表示されます。「アクション」の目のアイコンを押すと、正しく貼り付けられたか確かめられます。</li>
<li>行の左端の「<b>ノートブックからのアクセス</b>」のスイッチを押してオンにします（✕ の表示からオンの表示に変わります）。</li>
<li>保存ボタンはありません。入力した内容がそのまま保存されるので、シークレットの画面は閉じてかまいません。</li>
<li>上の手順 4 で「API で自動実行」を選び、手順 5 を実行します。シークレットへのアクセスを許可するか聞かれたら、許可します。</li>
</ol>
<p>シークレットの画面にある「Gemini API キー」は Google の別の AI 用なので、使いません。</p>
<p>手順 5 で「シークレット ANTHROPIC_API_KEY を読めませんでした」と出たときは、名前の綴りと「ノートブックからのアクセス」がオンになっているかを確認してください。
キーが間違っている・無効になっているときは、変換レポートの「AI による補完」の章にエラーが表示されます。</p>

<h4>3. キーを安全に扱う</h4>
<ul>
<li>キーをノートブックのセルやメール・チャットに直接書かないでください。シークレットに保存した値は非公開で、自分と、アクセスを許可したノートブックからしか見えません。ノートブックを他の人と共有しても、キーは渡りません。</li>
<li>キーが漏れたおそれがあるときは、Claude Console の「Settings → API keys」でそのキーを削除し、新しく作り直してください。</li>
</ul>

<h4>4. 料金の目安</h4>
<ul>
<li>「API で自動実行」で手順 5 を実行するたびに料金がかかります（既定のモデル <code>claude-opus-5-5</code> は、入力 100 万トークンあたり 4 ドル、出力 100 万トークンあたり 20 ドル）。</li>
<li>試験に使った論文（AI への依頼文が約 4 万字）では、1 回あたりおおむね 0.3〜1 ドル程度と見込まれます（論文の長さ・図の数で変わります）。</li>
<li>実際に使ったトークン数は、変換レポートの「AI による補完」の章の最後に表示されます。</li>
</ul>

<h4>5. API で使うモデルについて</h4>
<ul>
<li>手順 4 の「API で使うモデル（MODEL）」の既定は、このツールで動作を確かめた <code>claude-opus-5-5</code>（Claude Opus 5.5）です。
Anthropic の発表では、このモデルの提供終了は 2027 年 9 月 22 日より前にはならない予定です（<a href="https://platform.claude.com/docs/en/about-claude/model-deprecations" target="_blank">モデルの提供終了の予定</a>）。</li>
<li>将来このモデルが使えなくなった場合は、ツールが Anthropic のモデル一覧から使える最新の Opus を自動で選んでやり直し、変換レポートの「AI による補完」の章にその旨を表示します。</li>
<li>はじめから最新の Opus を使いたいときは、「自動（最新の Opus）」を選びます。ただし、動作を確かめていない新しいモデルが選ばれることがあります。</li>
<li>モデル名を自分で入力することもできます。使えるモデルの名前（Claude API ID）は、Anthropic のモデル一覧（<a href="https://platform.claude.com/docs/en/models/overview" target="_blank">Models overview</a>）で確認できます。</li>
<li>この「API で自動実行」は Claude（Anthropic）専用です。ChatGPT や Gemini の API キーは使えません。
ChatGPT や Gemini を使いたいときは「手動（チャットに貼り付け）」を選び、それぞれのチャット画面に依頼文を貼り付けてください（API キーは要りません）。</li>
</ul>
</details>

In [ ]:
#@title 5. 変換を実行
_need = [n for n in (1, 2, 3, 4,) if n not in globals().get("DONE", set())]
if _need:
    raise SystemExit("先に手順 " + "・".join(map(str, _need)) + " を実行してください（上から順に ▶ を押します。"
                     "前の手順をやり直したときは、その後の手順もやり直してください）。")
import shutil
from IPython.display import HTML, Markdown, display
from jasme_jats.colab import ai_required_box, error_box, no_ai_box, prompt_box, unplaced_box
from jasme_jats.config import ArticleMeta, JournalConfig
from jasme_jats.pipeline import AIRequiredError, Options, convert

api_key = None
if AI == "api":
    from google.colab import userdata
    try:
        api_key = userdata.get("ANTHROPIC_API_KEY")
    except Exception as e:
        raise SystemExit(f"シークレット ANTHROPIC_API_KEY を読めませんでした: {e}")

if CONVERT_EMF and globals().get("LO_PROC") is not None:
    if LO_PROC.poll() is None:
        print("LibreOffice のインストールが終わるのを待っています（EMF/WMF の図の変換に使います）…", flush=True)
        LO_PROC.wait()
    if not (shutil.which("soffice") or shutil.which("libreoffice")):
        print("※ LibreOffice をインストールできませんでした。EMF/WMF の図は変換されません（レポート 9 章）。記録:", LO_LOG)

OUT = WORK / "output"
shutil.rmtree(OUT, ignore_errors=True)
journal = JournalConfig.load()
meta = ArticleMeta.from_dict(meta_input)
try:
    res = convert(docx_path, meta, journal, Options(out_dir=OUT, ai_mode=AI, ai_model="auto" if MODEL.startswith("自動") else MODEL.strip(),
                                                  ai_api_key=api_key, pdf_path=pdf_path, split_labels=SPLIT_LABELS))
except AIRequiredError as e:
    display(HTML(ai_required_box(str(e))))
    raise SystemExit("手順 4 で「API で自動実行」か「手動（チャットに貼り付け）」を選び、手順 4 からやり直してください。")
fid = res.fidelity
print("文字の不変性チェック:", fid.summary(), f"（段落 {fid.units} 件中 {fid.mapped} 件を出力、{len(fid.excluded)} 件は除外）")
print("最終出力の再チェック:", "合格" if fid.final_ok else "不合格")
print("DTD 検証:", "合格" if res.dtd_valid else "不合格")
print("J-STAGE のアップロード時チェック:", "問題なし" if not res.jstage_issues else f"要対応 {len(res.jstage_issues)} 件")
for mid, msg in res.jstage_issues:
    print("  ", mid, msg)
manual_prompt = None
if AI == "none":
    display(HTML(no_ai_box()))
if AI == "manual" and res.prompt_path:
    # 依頼文は、レポートの前に目立つ枠で出し、レポートの後ろにも同じものをコピーできる枠を出す
    attach = OUT / "AI添付画像"
    image_names = sorted(p.name for p in attach.iterdir()) if attach.exists() else []
    for n in image_names:
        files.download(str(attach / n))
    manual_prompt = res.prompt_path.read_text(encoding="utf-8")
    display(HTML(prompt_box(manual_prompt, image_names)))
if res.ai.error:
    display(HTML(error_box("AI による補完ができませんでした", res.ai.error)))
if AI == "api":
    display(HTML(unplaced_box(res.plan)))
display(Markdown(res.report_path.read_text(encoding="utf-8")))
if manual_prompt:
    display(HTML(prompt_box(manual_prompt, compact=True)))
DONE = {n for n in globals().get("DONE", set()) if n < 5} | {5}

### 6.（手動モードのときだけ）AI の回答を貼り付けて再変換

チャットから返ってきた回答は、`json` と書かれたコードブロック（灰色の枠）1 つのはずです。枠の右上の**コピーボタン**でコピーし、
下のセルの `r'''` と `'''` の間に貼り付けてから ▶ を押してください（コードブロックの外の文章が混じらず、中身がそのままコピーされます）。

- 回答がコードブロックになっていない、またはアーティファクト（別の画面）として返ってきたときは、チャットで
  「回答の書き方に従って、```json のコードブロック 1 つで出し直してください」と頼んでください。
- 読めなかったときは、何行目のどこが読めないかが表示されます。

In [ ]:
AI_RESPONSE = r'''
ここに AI の回答を貼り付ける
'''

_need = [n for n in (1, 2, 3, 4, 5,) if n not in globals().get("DONE", set())]
if _need:
    raise SystemExit("先に手順 " + "・".join(map(str, _need)) + " を実行してください（上から順に ▶ を押します。"
                     "前の手順をやり直したときは、その後の手順もやり直してください）。")
if AI != "manual":
    raise SystemExit("この手順は手動モードのときだけ使います。手順 4 で「手動（チャットに貼り付け）」を選び、手順 5 からやり直してください。")
if not AI_RESPONSE.strip() or "ここに AI の回答を貼り付ける" in AI_RESPONSE:
    raise SystemExit("AI の回答が貼り付けられていません。上の r''' と ''' の間に貼り付けてから実行してください。")
res = convert(docx_path, meta, journal, Options(out_dir=OUT, ai_mode="manual", ai_response=AI_RESPONSE,
                                              pdf_path=pdf_path, split_labels=SPLIT_LABELS))
fid = res.fidelity
print("文字の不変性チェック:", fid.summary(), "/ 最終出力の再チェック:", "合格" if fid.final_ok else "不合格",
      "/ DTD 検証:", "合格" if res.dtd_valid else "不合格",
      "/ J-STAGE のアップロード時チェック:", "問題なし" if not res.jstage_issues else f"要対応 {len(res.jstage_issues)} 件")
if res.ai.error:
    display(HTML(error_box("貼り付けた回答を読み込めませんでした（AI による補完は反映されていません）", res.ai.error)))
display(HTML(unplaced_box(res.plan)))
display(Markdown(res.report_path.read_text(encoding="utf-8")))

In [ ]:
#@title 7. 結果をダウンロード
#@markdown 「変換結果.zip」の中に、J-STAGE にアップロードする jasme.zip、XML、変換レポートが入っています。
_need = [n for n in (1, 2, 3, 4, 5,) if n not in globals().get("DONE", set())]
if _need:
    raise SystemExit("先に手順 " + "・".join(map(str, _need)) + " を実行してください（上から順に ▶ を押します。"
                     "前の手順をやり直したときは、その後の手順もやり直してください）。")
archive = shutil.make_archive(str(WORK / "変換結果"), "zip", OUT)
files.download(archive)